In [ ]:
import pandas as pd
import numpy as np

In [3]:
import mysql.connector

try:
    conn = mysql.connector.connect(
        host="127.0.0.1",
        user='root',
        password="",
        database="project"
    )
    mycourser = conn.cursor()
    print("Connect to MySQL database")
except Exception as e:
    print('Error : ', e)

Connect to MySQL database


In [4]:
# Query execute karein
mycourser.execute("select * from retailcleaned")

# Result fetch karein
retail = mycourser.fetchall()  # Ya fetchone()

df = pd.DataFrame(retail)
df.to_csv("retail.csv", index=False)

In [10]:
df.columns = ['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer_ID', 'Country', 'SourceSheet']
df['InvoiceDate'] = pd.DatetimeIndex(df['InvoiceDate'])
df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer_ID,Country,SourceSheet
0,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01,1.25,13085.0,United Kingdom,2010-2011
1,489434,21523,FANCY FONT HOME SWEET HOME DOORMAT,10,2009-12-01,5.95,13085.0,United Kingdom,2010-2011
2,489434,21871,SAVE THE PLANET MUG,24,2009-12-01,1.25,13085.0,United Kingdom,2010-2011
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01,2.10,13085.0,United Kingdom,2010-2011
4,489434,22064,PINK DOUGHNUT TRINKET POT,24,2009-12-01,1.65,13085.0,United Kingdom,2010-2011


In [11]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1055236 entries, 0 to 1055235
Data columns (total 9 columns):
 #   Column       Non-Null Count    Dtype         
---  ------       --------------    -----         
 0   Invoice      1055236 non-null  object        
 1   StockCode    1055236 non-null  object        
 2   Description  1050854 non-null  object        
 3   Quantity     1055236 non-null  int64         
 4   InvoiceDate  1055236 non-null  datetime64[ns]
 5   Price        1055236 non-null  object        
 6   Customer_ID  812366 non-null   float64       
 7   Country      1055236 non-null  object        
 8   SourceSheet  1055236 non-null  object        
dtypes: datetime64[ns](1), float64(1), int64(1), object(6)
memory usage: 72.5+ MB


In [24]:
df['Price'] = df['Price'].astype('float')

In [106]:
rfm_data = df[(df['Customer_ID'].notna()) & (df['Quantity'] > 0)].copy()
snapshot_date = pd.to_datetime("2011-12-10")

rfm = rfm_data.groupby('Customer_ID').agg(
    recency = ('InvoiceDate', lambda x : (snapshot_date - x.max()).days),
    freqency = ('Invoice', 'nunique'),
    monetary = ('Quantity', lambda x : (x * rfm_data.loc[x.index, 'Price']).sum())
).reset_index()

In [114]:
rfm.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5881 entries, 0 to 5880
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Customer_ID  5881 non-null   float64
 1   recency      5881 non-null   int64  
 2   freqency     5881 non-null   int64  
 3   monetary     5881 non-null   float64
 4   r_score      5881 non-null   int64  
dtypes: float64(2), int64(3)
memory usage: 229.9 KB


In [121]:
rfm['r_score'] = pd.qcut(rfm['recency'], 5, labels=[5, 4, 3, 2, 1]).astype(int)
rfm['f_score'] = pd.qcut(rfm['freqency'].rank(method='first'), 5, labels=[1, 2, 3, 4, 5]).astype(int)
rfm['m_score'] = pd.qcut(rfm['monetary'].rank(method='first'), 5, labels=[1, 2, 3, 4, 5]).astype(int)

def segment_cus(row):
    if row['r_score'] >= 4 and row['f_score'] >= 4 and row['m_score'] >= 5:
        return "champions"

    if row['r_score'] >= 3 and row['f_score'] >= 3:
        return "loyal"

    if row['r_score'] <= 2 and row['f_score'] >= 3:
        return "at_risk"

    if row['r_score'] >= 4 and row['f_score'] <= 2:
        return "New"

    if row['r_score'] == 3 and row['m_score'] <= 3:
        return "need attention"

    else:
        return "lost"


rfm['segment'] = rfm.apply(segment_cus, axis=1)
rfm['IsChurned'] = rfm['recency'] > 90

In [125]:
df.columns

Index(['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate',
       'Price', 'Customer_ID', 'Country', 'SourceSheet'],
      dtype='object')

In [138]:
df = df[(df['Customer_ID'].notna()) & (df['Quantity'] > 0)].copy()
df['InvoiceMonth'] = df['InvoiceDate'].dt.to_period('M').dt.to_timestamp()
df['Revenue'] = df['Quantity'] * df['Price']

df_final = df.merge(rfm, how="left", on="Customer_ID")
df_final.to_excel("retailfinal.xlsx", index=False)